# 03 - Heterogeneous treatment effects

An average treatment effect answers "does this work?". It does not answer "who
should get it?", and those questions can have opposite answers: a programme with
a modest average can be strongly beneficial for one group and useless for
another.

Estimating effects that vary by unit is harder than estimating an average, in a
way that is easy to underestimate. This notebook shows one estimator failing so
completely that it reports *no* heterogeneity at all — and the reason has
nothing to do with the data.

## Causal question

An intervention is offered to patients. Its effect is believed to depend on
baseline risk. For which patients is it worth offering, and how much does the
benefit vary across them?

## Data and design

- **Unit of analysis:** one patient.
- **Treatment:** `treatment`, binary, with take-up depending on covariates.
- **Outcome:** `outcome`, continuous.
- **Covariates:** `age`, `risk_score`, `prior_usage`.
- **Ground truth:** `true_ite` carries each patient's actual individual effect,
  and `high_risk` flags the subgroup the generator treats differently.

Knowing `true_ite` is what makes this a lab rather than an analysis. It is the
only way to check whether an estimated CATE surface reflects reality or the
estimator's own assumptions.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor

from causal_inference_lab.data_generators import make_heterogeneous_treatment_data
from causal_inference_lab.estimators import aipw_ate, difference_in_means
from causal_inference_lab.meta_learners import SMetaLearner, TMetaLearner, XMetaLearner

COVARIATES = ["age", "risk_score", "prior_usage"]

dataset = make_heterogeneous_treatment_data(n=4_000, seed=123)
data = dataset.data
features = data[COVARIATES]

print(f"patients:         {len(data):,}")
print(f"treated share:    {data['treatment'].mean():.1%}")
print(f"true ATE:         {dataset.true_ate:.3f}")
print()
print("individual effects vary substantially:")
print(f"  sd:    {data['true_ite'].std():.3f}")
print(f"  range: {data['true_ite'].min():.3f} to {data['true_ite'].max():.3f}")
print()
print(data.groupby("high_risk")["true_ite"].agg(["count", "mean", "std"]).to_string(
    float_format=lambda v: f"{v:.3f}"))

**Interpretation.** The average effect of 1.38 hides a factor-of-two difference
between subgroups: 1.00 for the low-risk majority against 2.20 for the 31% at
high risk. A decision made on the average alone would treat these two groups
identically, which is the failure this notebook is about.

## Estimand

The **conditional average treatment effect (CATE)**: the expected effect for
patients with a given covariate profile.

The ATE is a single number; the CATE is a function. Reporting a CATE surface
commits you to a claim at every point on it, which is a much stronger claim than
an average and deserves correspondingly more scepticism.

## Identification assumptions

The assumptions are the same as for any selection-on-observables estimator, and
they must now hold *locally* rather than on average.

1. **Conditional ignorability.** Treatment is as good as random given the
   covariates.
2. **Overlap, everywhere it matters.** Not just overall: each region of
   covariate space where a CATE is reported needs both treated and untreated
   units. A subgroup that is almost always treated has no local comparison.
3. **The estimator can represent the true effect surface.** This is not usually
   listed as an identification assumption, and the first result below shows why
   it belongs here.

Assumption 3 is the one this notebook is designed to expose.

## Estimation

First the ATE, as a baseline, and then three meta-learners with their default
linear models.

- **S-learner:** one model on all data, with treatment as a feature.
- **T-learner:** separate models per arm, CATE as the difference in predictions.
- **X-learner:** imputes individual effects, then models those.

In [ ]:
print(f"naive difference in means: {difference_in_means(data).estimate:.3f}")
print(f"AIPW:                      {aipw_ate(data, covariates=COVARIATES).estimate:.3f}")
print(f"true ATE:                  {dataset.true_ate:.3f}")


def evaluate(learner, label: str) -> dict[str, float]:
    """Fit a learner and score its CATE surface against the known truth."""
    learner.fit(data, covariates=COVARIATES, treatment_col="treatment", outcome_col="outcome")
    predicted = learner.predict_cate(features)
    truth = data["true_ite"].to_numpy()
    return {
        "learner": label,
        "mean CATE": float(np.mean(predicted)),
        "sd CATE": float(np.std(predicted)),
        "corr with truth": float(np.corrcoef(predicted, truth)[0, 1]),
        "RMSE": float(np.sqrt(np.mean((predicted - truth) ** 2))),
    }


linear_results = pd.DataFrame(
    [
        evaluate(SMetaLearner(), "S-learner (linear)"),
        evaluate(TMetaLearner(), "T-learner (linear)"),
        evaluate(XMetaLearner(), "X-learner (linear)"),
    ]
)
print()
print(f"true effect sd for comparison: {data['true_ite'].std():.3f}")
print(linear_results.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

**Interpretation.** The S-learner reports a CATE standard deviation of **0.000**
and a correlation with the truth of −0.015. It has not estimated a weak
heterogeneity; it has estimated none. Every patient receives the same predicted
effect of 1.49.

The cause is structural, not statistical. A linear model with treatment as an
additive feature has no term in which treatment interacts with a covariate, so
its predicted effect is the treatment coefficient — a constant, for everyone, by
construction. No sample size fixes this. The model answered a question about
heterogeneity by assuming there is none, and reported the assumption as a
finding.

The T- and X-learners recover the structure well (correlation 0.80), because
fitting the arms separately lets the two response surfaces differ. Here they
agree with each other to three decimal places.

The S-learner's failure was the *model*, not the *design*. Giving the same
learner a model that can express interactions should fix it entirely.

In [ ]:
flexible_results = pd.DataFrame(
    [
        evaluate(
            SMetaLearner(model=GradientBoostingRegressor(random_state=0)),
            "S-learner (boosted)",
        ),
        evaluate(
            TMetaLearner(
                treated_model=GradientBoostingRegressor(random_state=0),
                control_model=GradientBoostingRegressor(random_state=0),
            ),
            "T-learner (boosted)",
        ),
    ]
)
print(pd.concat([linear_results, flexible_results]).to_string(
    index=False, float_format=lambda v: f"{v:.3f}"))

**Interpretation.** Two reversals.

The boosted S-learner goes from the worst estimator to the best: correlation
0.93 and RMSE 0.24, beating every alternative. The architecture was never the
problem.

The boosted T-learner goes the other way — RMSE rises from 0.39 to 0.46, and its
CATE spread of 0.78 overshoots the true 0.60. Fitting flexible models on half
the data each costs more in variance than the extra flexibility returns. "Use a
more powerful model" is not a direction that always improves things; it trades
bias for variance, and the T-learner was already paying a variance price by
splitting the sample.

## Diagnostics

For a CATE surface the diagnostic question is not "is the average right" but
"would this ranking survive being re-estimated". We check stability by fitting
each learner on two disjoint halves and comparing the predictions they make
about the *same* patients.

In [ ]:
rng = np.random.default_rng(0)
shuffled = rng.permutation(len(data))
first_half, second_half = shuffled[: len(data) // 2], shuffled[len(data) // 2 :]


def split_half_agreement(make_learner, label: str) -> None:
    """Fit on two disjoint halves, correlate the two CATE surfaces."""
    predictions = []
    for rows in (first_half, second_half):
        learner = make_learner()
        learner.fit(
            data.iloc[rows],
            covariates=COVARIATES,
            treatment_col="treatment",
            outcome_col="outcome",
        )
        predictions.append(learner.predict_cate(features))
    agreement = np.corrcoef(predictions[0], predictions[1])[0, 1]
    print(f"{label:22s} split-half correlation: {agreement:.3f}")


split_half_agreement(TMetaLearner, "T-learner (linear)")
split_half_agreement(
    lambda: SMetaLearner(model=GradientBoostingRegressor(random_state=0)),
    "S-learner (boosted)",
)

**Interpretation.** The linear T-learner is almost perfectly reproducible
(0.994) — unsurprising, since a linear surface has few parameters to disagree
about. The boosted S-learner is noticeably less stable (0.845) despite being
more *accurate* against the truth.

This is the trade-off that matters for decisions. The boosted model recovers the
real surface better, but a meaningful part of what it fits is specific to the
sample it saw. Had we no ground truth — the normal situation — stability alone
would have pointed at the wrong model.

## Uncertainty

The practical use of a CATE surface is ranking: treat the patients with the
largest predicted benefit. So the honest uncertainty question is how much of the
achievable gain that ranking actually captures.

In [ ]:
truth = data["true_ite"].to_numpy()
budget = len(data) // 5  # capacity to treat 20% of patients
ideal = set(np.argsort(-truth)[:budget])


def targeting_quality(learner, label: str) -> None:
    """How good is the top-20% chosen by this learner's ranking?"""
    learner.fit(data, covariates=COVARIATES, treatment_col="treatment", outcome_col="outcome")
    chosen = np.argsort(-learner.predict_cate(features))[:budget]
    print(
        f"{label:22s} overlap with ideal: {len(set(chosen) & ideal) / budget:.1%}   "
        f"true effect in chosen group: {truth[chosen].mean():.3f}"
    )


print(f"treating everyone would average:  {truth.mean():.3f}")
print(f"the best possible top-20% averages: {truth[list(ideal)].mean():.3f}")
print()
targeting_quality(TMetaLearner(), "T-learner (linear)")
targeting_quality(SMetaLearner(model=GradientBoostingRegressor(random_state=0)), "S-learner (boosted)")
targeting_quality(SMetaLearner(), "S-learner (linear)")

**Interpretation.** Both working learners select a group whose true average
effect is around 2.23 against a population average of 1.38 — roughly 60% more
benefit per patient treated, from the same budget. They agree with the ideal
selection on about three-quarters of patients, and the quarter they miss costs
little, because patients near the cutoff have similar effects by definition.

The linear S-learner is the instructive case: its predictions are constant, so
its "ranking" is arbitrary tie-breaking. It overlaps the ideal selection on
24.4% of patients — close to the 20% a coin flip would achieve — and the group
it picks averages 1.459 against a population 1.377. Nearly the entire targeting
gain is gone.

An estimator that silently reports no heterogeneity does not merely lose
accuracy. It destroys the value of targeting while producing output that looks
like a perfectly normal result: a mean CATE of 1.49 is a plausible number, and
nothing but the zero standard deviation gives the failure away.

## Limitations

- **Ground truth is a luxury of synthetic data.** Every validation here rests on
  `true_ite`. In a real application none of these correlations or RMSEs are
  computable, and the model choice must be made on stability and holdout fit
  alone — which, as the diagnostics showed, can point the wrong way.
- **CATE estimates are far noisier than ATE estimates.** The split-half
  correlations quantify that. Treating a predicted individual effect as a
  reliable statement about one patient is not supported.
- **No confidence intervals on the CATE surface.** The learners return point
  predictions. Honest CATE inference needs bootstrapping or an
  uncertainty-quantifying learner; neither is implemented here.
- **Overlap was not checked per region.** The identification assumptions demand
  local overlap wherever a CATE is reported. This notebook checks it nowhere,
  which is a gap, not an oversight worth hiding.
- **Learners use untuned defaults.** Boosting hyperparameters were not
  cross-validated, and the comparison would shift if they were.
- **Targeting assumes the ranking generalises.** The chosen group is evaluated
  on the same data that produced the ranking, which flatters it.